# SmartDecline PCMCI 再検証 v3 — ローカル Jupyter（並列）

セル 1〜3 を順に実行 → セル 4 でワーカー起動 → セル 5 で進捗確認 → 全終了後にセル 6・7・8。

中断・再開：同じ `OUT` で同じセルを再実行すれば、済んだ反復はチェックポイントから読み、未完了分だけ計算します。モジュールを差し替えた場合は `OUT` を別名にしてください。

In [ ]:
# ── セル 1：パスと共通設定 ──────────────────────────────────────────
from pathlib import Path
import os, sys, subprocess, shutil, threading, time, glob, json
import pandas as pd

WORKDIR = Path(r'G:\マイドライブ\13_SmartDecline\73_結果（DBN）\25_PCMCI相関係数_20260929')
XLSX    = WORKDIR / '11_満足度・生活の質に関する調査_20192025_20260927.xlsx'
MODULE  = WORKDIR / 'validation_analysis.py'
OUT     = Path.home() / 'smartdecline_validation'     # ローカル出力（Drive 同期を避ける）

QUICK   = False            # True: 動作確認（少数反復）。研究結果には使わない
NREP    = 3 if QUICK else 500
NBOOT   = 30 if QUICK else 2000
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)            # 1 コアは OS/Jupyter 用に残す
if QUICK: OUT = OUT.with_name(OUT.name + '_smoke')

assert WORKDIR.is_dir(), WORKDIR
assert XLSX.is_file(), XLSX
assert MODULE.is_file(), MODULE
os.chdir(WORKDIR)
OUT.mkdir(parents=True, exist_ok=True)

BASE = [sys.executable, '-u', str(MODULE), '--xlsx', str(XLSX), '--outdir', str(OUT),
        '--seed', '2026', '--window', 'lag1']
ENV  = dict(os.environ, OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
print('python :', sys.executable)
print('module :', MODULE)
print('data   :', XLSX)
print('output :', OUT)
print('workers:', N_WORKERS, '| NREP', NREP, '| NBOOT', NBOOT)


In [ ]:
# ── セル 2：依存パッケージの確認（未導入なら入れる）───────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tigramite>=5.2', 'numpy', 'pandas', 'scipy', 'openpyxl'], check=True)
import importlib.metadata as m
print({p: m.version(p) for p in ('tigramite', 'numpy', 'pandas', 'scipy')})


In [ ]:
# ── セル 3：速いステップを 1 プロセスで（Gate 1・V1・V2・V4・V5・V7・V8、数分）────
# ここで run_metadata.json と対象モデルのキャッシュが作られ、以降のワーカーが安全に並走できる
r = subprocess.run(BASE + ['--steps', 'gate,diag,common,boot,ahiv,window_check,period_check', '--nboot', str(NBOOT)],
                   env=ENV, capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, 'fast steps failed'


In [ ]:
# ── セル 4：置換検定をワーカーに分けてバックグラウンド起動 ─────────────
# 負荷単位: V3 は 1 反復 = 3 モデル、V6 は 1 反復 = 2 モデル。V3 を 3 分割、V6 を 2 分割すると各タスク ≈ 500 モデル。
def slices(n, k):
    edges = [round(i * n / k) for i in range(k + 1)]
    return [(edges[i], edges[i + 1]) for i in range(k)]

tasks = []
for scale in ('SMC', 'Large'):
    for a, b in slices(NREP, 3):
        tasks.append((f'V3_{scale}_{a}-{b}',
                      ['--steps', 'perm_time', '--nrep-time', str(NREP), '--scales', scale, '--rep-range', f'{a}:{b}']))
    for mode in ('global', 'region'):
        for a, b in slices(NREP, 2):
            tasks.append((f'V6_{scale}_{mode}_{a}-{b}',
                          ['--steps', 'perm_label', '--nrep-label', str(NREP), '--scales', scale,
                           '--label-blocks', mode, '--rep-range', f'{a}:{b}']))
print(len(tasks), 'tasks;', N_WORKERS, 'concurrent workers')

STATE = {'running': {}, 'done': [], 'failed': [], 'pending': list(tasks), 'finished': False}

def scheduler():
    while STATE['pending'] or STATE['running']:
        for name, p in list(STATE['running'].items()):
            rc = p.poll()
            if rc is not None:
                STATE['running'].pop(name)
                (STATE['done'] if rc == 0 else STATE['failed']).append(name)
        while STATE['pending'] and len(STATE['running']) < N_WORKERS:
            name, args = STATE['pending'].pop(0)
            log = open(OUT / f'worker_{name}.log', 'w', encoding='utf-8')
            STATE['running'][name] = subprocess.Popen(BASE + args, env=ENV, stdout=log, stderr=subprocess.STDOUT)
            time.sleep(2)                     # 起動をずらす（同時初期化の回避）
        time.sleep(5)
    STATE['finished'] = True

threading.Thread(target=scheduler, daemon=True).start()
print('started; monitor with the next cell')


In [ ]:
# ── セル 5：進捗モニター（随時実行）─────────────────────────────────
print(f"running {len(STATE['running'])} | pending {len(STATE['pending'])} | done {len(STATE['done'])} | failed {len(STATE['failed'])} | finished {STATE['finished']}")
for f in sorted(glob.glob(str(OUT / 'worker_*.log'))):
    lines = [l for l in open(f, encoding='utf-8').read().splitlines() if l.strip()]
    tail = next((l for l in reversed(lines) if l.startswith('[V')), lines[-1] if lines else '(starting)')
    print(f'{Path(f).stem[7:]:28s} {tail}')
if STATE['failed']:
    print('\n失敗したタスク:', STATE['failed'], '→ 該当ログを確認し、セル 4 を再実行（済んだ反復は再利用される）')


In [ ]:
# ── セル 6：集計（全ワーカー終了後に実行。チェックポイントを読むだけなので数十秒）────
assert STATE['finished'] and not STATE['failed'], 'ワーカーが未完了または失敗しています（セル 5 を確認）'
for scale in ('SMC', 'Large'):
    for args in (['--steps', 'perm_time', '--nrep-time', str(NREP), '--scales', scale],
                 ['--steps', 'perm_label', '--nrep-label', str(NREP), '--scales', scale, '--label-blocks', 'global,region']):
        r = subprocess.run(BASE + args, env=ENV, capture_output=True, text=True)
        print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-3000:]
print('aggregated')


In [ ]:
# ── セル 7：結果の要約 ─────────────────────────────────────────────
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_colwidth', 60)
for f in ['G1_table1_check.csv', 'V1_diagnostics.csv', 'V2_common_set.csv', 'V4_cluster_boot.csv',
          'V5_anderson_hsiao.csv', 'V7_window_sensitivity.csv', 'V8_period_position_contrasts.csv']:
    p = OUT / f
    if p.exists():
        print('=====', f); display(pd.read_csv(p).round(3))

for f in sorted(glob.glob(str(OUT / 'V3_perm_time_*.json')) + glob.glob(str(OUT / 'V6_perm_label_*.json'))):
    s = json.load(open(f, encoding='utf-8')); print('=====', Path(f).name)
    print('observed:', {k: round(v, 3) for k, v in s['observed'].items() if isinstance(v, float)})
    if 'blocks' in s: print('blocks:', s['blocks'])
    for k, v in s.items():
        if '__p_' in k or k.startswith('p_') or k.startswith('null_mean') or k.startswith('null_share'):
            print(f'  {k} = {v:.3f}')

for f in sorted(glob.glob(str(OUT / 'V3_table6_edges_vs_null_*.csv'))):
    d = pd.read_csv(f); print('=====', Path(f).name, '(reported edges only)'); display(d[d.reported].round(3))


In [ ]:
# ── セル 8：結果だけを Drive にコピー（チェックポイントとログは除外）────────
dest = WORKDIR / OUT.name
shutil.copytree(OUT, dest, ignore=shutil.ignore_patterns('checkpoints', 'worker_*.log'), dirs_exist_ok=True)
print('copied to', dest)
print(sorted(p.name for p in dest.iterdir()))
